# Demonstrating dynamical decoupling fidelity gains

A hardware-ready DD study in three parts, modeled on the literature:

1. **Single-qubit state preservation, sequence comparison** — survival of
   a superposition state over an idle-time sweep, free evolution vs CPMG,
   XY4, and XY8 at fixed pulse interval. This is the protocol of
   **Pokharel, Anand, Fortman & Lidar, PRL 121, 220502 (2018)**
   ([arXiv:1807.08768](https://arxiv.org/abs/1807.08768)), with the
   sequence menu of the survey by Ezzell et al.
   ([arXiv:2207.03670](https://arxiv.org/abs/2207.03670)).
2. **Entanglement preservation** — the same sweep for a Bell pair, DD on
   both qubits.
3. **Spectator DD during gates** — decoupling an idle qubit while a
   neighboring pair runs a CX train. This is how DD is used in IBM's
   Quantum Volume 64 demonstration
   ([arXiv:2008.08571](https://arxiv.org/abs/2008.08571)): not as a
   stand-alone experiment, but protecting idle qubits inside algorithmic
   circuits.

Each part builds and **preflights its pulse schedules offline**; the
hardware runs are gated behind `RUN_ON_HARDWARE`. The setup (offline
`qubex.Experiment` from [qubex-config/](qubex-config/)) is the same as in
[tutorial.ipynb](tutorial.ipynb).

Requirements:

```bash
pip install "qiskit-qubex-provider[qubex,plot] @ git+https://github.com/orangekame3/qiskit-qubex-provider.git"
```

In [1]:
import numpy as np
import plotly.graph_objects as go
import yaml
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import XGate, YGate
from qubex import Experiment

from qiskit_qubex_provider import (
    QubexProvider,
    build_device_topology,
    build_dynamical_decoupling_pass_manager,
    build_pulse_schedule_timeline_figure,
    extract_pulse_timeline,
)

In [2]:
topology = build_device_topology(
    calib_note_path="qubex-config/calibration/calib_note.json",
    params_dir="qubex-config/params",
    name="4Q-DEMO",
    device_id="4Q-DEMO",
)
exp = Experiment(
    system_id="4Q-DEMO-SYS",
    muxes=[0],
    config_dir="qubex-config/config",
    params_dir="qubex-config/params",
    calib_note_path="qubex-config/calibration/calib_note.json",
    calibration_valid_days=10000,
    mock_mode=True,
)
backend = QubexProvider.from_experiment(exp, device_topology=topology).get_backend()

RUN_ON_HARDWARE = False  # flip on a connected setup (real Experiment, no mock_mode)
SHOTS = 1024

[qubex.experiment.experiment_context] WARNING: Skew file not found: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/examples/qubex-config/config/skew.yaml
date: 2026-06-12 21:59:24
python: 3.13.11
qubex: 1.5.0b4
env: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/.venv
config: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/examples/qubex-config/config
params: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/examples/qubex-config/params
chip: 16Q-DEMO (16-qubit demo chip)
qubits: ['Q00', 'Q01', 'Q02', 'Q03']
muxes: ['MUX0']
boxes: ['DEMO1', 'DEMO2']


## Part 1 — single-qubit state preservation, sequence comparison

Per idle time *T*: prepare |+⟩ (`h`), idle for *T* under one of the
decoupling strategies, undo the preparation, measure. Survival is
*P*(`"0"`).

By default the pass manager (like Qiskit's `PadDynamicalDecoupling`)
inserts **one** sequence block per idle window and stretches it, so the
pulse interval grows with *T*. The papers instead keep the pulse
interval fixed and repeat the sequence — pass `pulse_interval` and the
pass manager picks the repetition count **per idle window**: at every
*T* the qubit sees a π pulse every ≈250 ns, regardless of the sequence
family.

In [3]:
QUBIT = 0  # physical qubit Q00
IDLE_TIMES_US = [0.5, 1, 2, 4, 8, 16]
PULSE_INTERVAL_NS = 250

X, Y = XGate(), YGate()
SEQUENCES = {
    "cpmg": [X, X],
    "xy4": [X, Y, X, Y],
    "xy8": [X, Y, X, Y, Y, X, Y, X],
}


def idle_circuit(idle_ns: float) -> QuantumCircuit:
    qc = QuantumCircuit(4, 1)
    qc.h(QUBIT)
    qc.delay(idle_ns, QUBIT, unit="ns")
    qc.h(QUBIT)
    qc.measure(QUBIT, 0)
    return qc


def dd_pass(base: list, **kwargs):
    return build_dynamical_decoupling_pass_manager(
        backend,
        sequence=base,
        pulse_interval=PULSE_INTERVAL_NS * 1e-9,
        scheduling_method="asap",
        **kwargs,
    )


sweep = {"free": []}
for idle_us in IDLE_TIMES_US:
    sweep["free"].append(
        transpile(idle_circuit(idle_us * 1000), backend,
                  scheduling_method="asap", optimization_level=1)
    )
for name, base in SEQUENCES.items():
    sweep[name] = []
    for idle_us in IDLE_TIMES_US:
        idle_ns = idle_us * 1000
        physical = transpile(idle_circuit(idle_ns), backend, optimization_level=1)
        sweep[name].append(dd_pass(base).run(physical))

for name in sweep:
    ops = dict(sweep[name][3].count_ops())  # the T = 4 µs point
    print(f"{name:5s} @4µs:", {k: v for k, v in ops.items() if k in ('x', 'y')} or "no DD pulses")

free  @4µs: no DD pulses
cpmg  @4µs: {'x': 16}
xy4   @4µs: {'x': 8, 'y': 8}
xy8   @4µs: {'x': 8, 'y': 8}


Note the constant pulse *budget*: at *T* = 4 µs all three sequences place
16 π pulses, just in different patterns — CPMG repeats X–X (robust against
pure dephasing, sensitive to pulse errors along X), XY4 alternates axes
(first-order universal), XY8 adds time-symmetry (second-order).

### Pulse-level sanity check

Compile the *T* = 4 µs point and look before spending hardware time:

In [4]:
schedule_free = backend.validate(sweep["free"][3])[0]
build_pulse_schedule_timeline_figure(schedule_free, title="Free evolution, T = 4 µs").show()

In [5]:
schedule_xy4 = backend.validate(sweep["xy4"][3])[0]
build_pulse_schedule_timeline_figure(schedule_xy4, title="XY4, T = 4 µs").show()

In [6]:
for name in SEQUENCES:
    schedule = backend.validate(sweep[name][3])[0]
    starts = [e["start_ns"] for e in extract_pulse_timeline(schedule)["Q00"]
              if e["kind"] == "pulse"]
    print(f"{name:5s}: {len(starts)} pulses, interval ≈ {np.mean(np.diff(starts[1:-1])):.0f} ns")

cpmg : 18 pulses, interval ≈ 250 ns
xy4  : 18 pulses, interval ≈ 250 ns
xy8  : 18 pulses, interval ≈ 250 ns


In [7]:
n_schedules = sum(len(circuits) for circuits in sweep.values())
for circuits in sweep.values():
    backend.validate(circuits)
print(f"Part 1: {n_schedules} schedules build cleanly")

Part 1: 24 schedules build cleanly


### Run and analyze

On hardware this is one `backend.run` per strategy. The fit extracts an
effective decay time per curve — the paper-style headline number is the
ratio of DD-protected to free-evolution decay times.

In [8]:
survival_part1 = None
if RUN_ON_HARDWARE:
    def survival(circuits, key="0"):
        result = backend.run(circuits, shots=SHOTS).result()
        return [result.get_counts(i).get(key, 0) / SHOTS for i in range(len(circuits))]

    survival_part1 = {name: survival(circuits) for name, circuits in sweep.items()}
    print(survival_part1)

In [9]:
if survival_part1 is not None:
    from scipy.optimize import curve_fit

    def decay(t, t2, p):
        return 0.5 * (1 + np.exp(-((t / t2) ** p)))

    for name, values in survival_part1.items():
        (t2, p), _ = curve_fit(decay, IDLE_TIMES_US, values,
                               p0=[10, 1], bounds=([0.1, 0.5], [1000, 3]))
        print(f"{name:5s}: T2_eff = {t2:6.1f} µs (stretch exponent {p:.2f})")

### What to expect

Illustrative model curves (not measurements!): free evolution decays on a
short Gaussian dephasing scale, DD pushes the decay toward the echo limit
(*T₂ᵉᶜʰᵒ* from `qubex-config/params/t2_echo.yaml`). On real transmons the
ordering XY8 ≳ XY4 ≳ CPMG ≫ free is typical for |+⟩-type states, with the
differences between sequences much smaller than the gap to free evolution
(see the survey for the full picture). Measured points overlay
automatically once `RUN_ON_HARDWARE` produced them.

In [10]:
t2_echo_us = yaml.safe_load(open("qubex-config/params/t2_echo.yaml"))["data"]["Q00"]
t2_star_us = 0.3 * t2_echo_us  # illustrative free-dephasing scale

t = np.linspace(0.01, max(IDLE_TIMES_US), 300)
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=0.5 * (1 + np.exp(-((t / t2_star_us) ** 2))),
                         name="free evolution (model)", line=dict(color="#dc2626", dash="dot")))
fig.add_trace(go.Scatter(x=t, y=0.5 * (1 + np.exp(-(t / t2_echo_us))),
                         name="DD-protected (model)", line=dict(color="#2563eb", dash="dot")))
if survival_part1 is not None:
    colors = {"free": "#dc2626", "cpmg": "#16a34a", "xy4": "#2563eb", "xy8": "#9333ea"}
    for name, values in survival_part1.items():
        fig.add_trace(go.Scatter(x=IDLE_TIMES_US, y=values, mode="lines+markers",
                                 name=f"{name} (measured)",
                                 marker=dict(color=colors[name], size=9)))
fig.update_layout(
    title="Survival of |+⟩ vs idle time",
    xaxis=dict(title="Idle time T (µs)", type="log"),
    yaxis=dict(title="P(0)", range=[0.45, 1.02]),
    width=700, height=420,
)
fig.show()

## Part 2 — preserving entanglement

The same protocol for a Bell pair on the calibrated `Q00`–`Q01` coupling:
prepare |Φ⁺⟩ (`h`, `cx`), idle both qubits for *T*, undo the preparation,
and measure both. Ideal survival is *P*(`"00"`). Entangled states decay
faster than single-qubit ones (both qubits dephase), which makes the DD
gain more dramatic — and protecting them is the realistic use case.

In [11]:
BELL_TIMES_US = [1, 2, 4, 8]


def bell_circuit(idle_ns: float) -> QuantumCircuit:
    qc = QuantumCircuit(4, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.delay(idle_ns, 0, unit="ns")
    qc.delay(idle_ns, 1, unit="ns")
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    return qc


bell = {"free": [], "xy4": []}
for idle_us in BELL_TIMES_US:
    idle_ns = idle_us * 1000
    bell["free"].append(
        transpile(bell_circuit(idle_ns), backend,
                  scheduling_method="asap", optimization_level=1)
    )
    physical = transpile(bell_circuit(idle_ns), backend, optimization_level=1)
    bell["xy4"].append(dd_pass(SEQUENCES["xy4"]).run(physical))

backend.validate(bell["free"] + bell["xy4"])
print(f"Part 2: {len(bell['free']) + len(bell['xy4'])} schedules build cleanly")

Part 2: 8 schedules build cleanly


In [12]:
schedule_bell = backend.validate(bell["xy4"][2])[0]  # T = 4 µs
build_pulse_schedule_timeline_figure(
    schedule_bell, title="Bell pair with XY4 on both qubits, T = 4 µs"
).show()

Both drive channels carry the decoupling train between the entangling and
disentangling blocks. (By default the pulses on `Q00` and `Q01` are
simultaneous; staggering them — e.g. via the `spacing` knob or the
context-aware pass — additionally suppresses the static ZZ coupling
between neighbors, which is exactly the trick used at scale in the QV64
work.)

In [13]:
survival_bell = None
if RUN_ON_HARDWARE:
    survival_bell = {
        name: survival(circuits, key="00")
        for name, circuits in bell.items()
    }
    print(survival_bell)

## Part 3 — spectator DD while neighbors run gates

DD's day job in algorithmic circuits (the QV64 setting): a qubit holding
a superposition sits idle while its neighbors execute gates, and crosstalk
plus dephasing degrade it. Here `q0` holds |+⟩ while `q3`–`q2` run a
train of three CXs; we compare leaving `q0` alone against decoupling it
(`qubits=[0]` restricts the DD pass to the spectator):

In [14]:
def spectator_circuit() -> QuantumCircuit:
    qc = QuantumCircuit(4, 1)
    qc.h(0)
    for _ in range(3):
        qc.cx(3, 2)
        qc.barrier(2, 3)  # keep the CXs from cancelling
    qc.barrier()
    qc.h(0)
    qc.measure(0, 0)
    return qc


spectator_free = transpile(spectator_circuit(), backend,
                           scheduling_method="asap", optimization_level=1)
physical = transpile(spectator_circuit(), backend, optimization_level=1)
spectator_dd = build_dynamical_decoupling_pass_manager(
    backend,
    sequence="xy4",
    pulse_interval=PULSE_INTERVAL_NS * 1e-9,
    scheduling_method="asap",
    qubits=[0],  # decouple only the spectator
).run(physical)

schedule_spectator = backend.validate(spectator_dd)[0]
build_pulse_schedule_timeline_figure(
    schedule_spectator, title="Spectator DD on Q00 during a CX train on Q03–Q02"
).show()

In [15]:
survival_spectator = None
if RUN_ON_HARDWARE:
    survival_spectator = {
        "free": survival([spectator_free])[0],
        "xy4": survival([spectator_dd])[0],
    }
    print(survival_spectator)

## Wrap-up

What to look for on hardware:

- **Part 1**: DD curves well above free evolution for *T* between a few
  and a few tens of microseconds; fitted *T₂* ratios quantify the gain.
  Sweep more initial states (|1⟩, |+i⟩, ...) — the paper shows the gain is
  state-dependent — and more sequences from the survey (UR, KDD, RGA).
- **Part 2**: the Bell survival gap should exceed the single-qubit one.
- **Part 3**: the spectator's survival with DD should stay near the
  isolated-qubit value even while the neighbors are driven.

See [dynamical-decoupling.ipynb](dynamical-decoupling.ipynb) for the
pass-manager basics and
[docs/dynamical-decoupling.md](../docs/dynamical-decoupling.md) for all
knobs (including the context-aware, staggered variant).